In [1]:
from utils import * 
import numpy as np
import shutil 
from matplotlib.lines import Line2D
import itertools
import os
import re
import sys 
from dataclasses import dataclass

%load_ext autoreload 
%autoreload 2

@dataclass
class Databases:
    input: str
    output_search: str
    output_align: str
    output_msa: str

In [29]:
msa_dir_path = '../data/genes/mmseqs/msa'

def get_msas_unpaired(gene_ids:list, dir_path:str=msa_dir_path):
    '''Load the unpaired MSAs for the specified gene IDs.
    
    :param gene_ids: The IDs of the genes to construct paired MSAs for. These should all be from the same genome.   
    :param dir_path: The path to the directory where the MSAs are stored. This function assumes the a3m files have names matching the gene ID of the query 
        sequence. 
    '''
    paths = {gene_id:os.path.join(dir_path, f'{gene_id}.a3m') for gene_id in gene_ids}
    return {gene_id:str(FASTAFile.from_file(path)) for gene_id, path in paths.items()}

# Custom MSAs

Analyzing the results of ColabFold structure prediction makes it clear that many of the Betazoid proteins have none (or very few) proteins populating their MSAs, and consequently have very weak folds. 

According to the [AlphaFold3 documentation](https://github.com/google-deepmind/alphafold3/blob/main/docs/installation.md), AlphaFold3 requires at least 3 of the following databases.
1. BFD
2.  MGnify
3. PDB 
4. UniProt
5. UniRef90

Although AlphaFold's JackHMMer-based homology search is unable to recover homologs for the Betazoid genes in these databases, there are likely far more homologs in ggKbase (particulary for the genes we already know are conserved amongst Betazoids). Here, we construct custom MSAs for select Betazoid proteins using ggKbase BLAST hits. 



In [ ]:
level_1_conserved_cluster_ids = [8, 7, 0, 4, 5, 6, 1, 3, 2] # Gene clusters found in > 7 Betazoids. 
level_2_conserved_cluster_ids = [9, 12, 10, 13, 11] # Gene clusters found in > 5 Betazoids
level_3_conserved_cluster_ids = [18, 42, 33, 25, 17, 19, 20, 21, 50, 31, 40, 27, 34, 15, 14, 54, 24, 23, 41, 39, 57, 47, 48, 51, 45, 35, 49, 28, 62, 29] # Gene clusters found in > 1 Betazoid.

conserved_cluster_ids = level_1_conserved_cluster_ids + level_2_conserved_cluster_ids + level_3_conserved_cluster_ids

genes_df = pd.read_csv('../data/genes/genes.csv', index_col=0)
genes_df['cluster_id'] = genes_df.index.map(json.load(open('../data/genes/clusters.json', 'r')))

filters = dict()
filters['all'] = np.array([True]*len(genes_df))
filters['prodigal'] = ~genes_df.prodigal_gene_id.isnull()
filters['conserved'] = genes_df.cluster_id.isin(conserved_cluster_ids)
for name, filter_ in filters.items():
    genes_df_ = genes_df[filter_]
    print(f'Number of genes passing filter {name}:', filter_.sum())
    FASTAFile.from_df(genes_df_).write(f'0-structures-custom_msa-genes_{name}.faa')


Number of genes passing filter all: 1633
Number of genes passing filter prodigal: 429
Number of genes passing filter conserved: 196


## BLASTp search against ggKbase

The ggKbase web iterface only allows one query sequence at a time. Even limiting the queries to genes in the conserved gene clusters, searching each individually would require nearly individual queries. 

*Instead of using the web-based, I asked Shufei to run the BLAST queries for me, and provide the corresponding amino acid sequences.I used lenient search parameters, with a maximum allowed E-value of 10.*

In [ ]:
genes_df = FASTAFile.from_file('0-structures-custom_msa-genes_prodigal.faa').to_df()
genes_df = genes_df[~genes_df.index.str.contains('bz_12')].copy()

In [4]:
blast_ggkbase_dir = '../data/genes/blast/ggkbase' # Directory where the BLAST search outputs are stored. 
blast_ggkbase_path = os.path.join(blast_ggkbase_dir, f'genes_prodigal.txt')
blast_ggkbase_df = BLASTFile.from_file(blast_ggkbase_path).to_df()

In [5]:
filters = dict()
filters['large_length_difference'] = np.abs(blast_ggkbase_df.query_length - blast_ggkbase_df.target_length) > 100
filters['low_query_coverage'] = np.abs(blast_ggkbase_df.alignment_length / blast_ggkbase_df.query_length) < 0.5
filters['bz_12_gene'] = blast_ggkbase_df.query_id.str.contains('bz_12')

# Odd that there are about 50 genes missing from the BLAST results, suggests that these were not called
# in the ggKbase Prodigal run. 
blast_ggkbase_df = apply_filters(filters, blast_ggkbase_df)

print('Number of query genes:', len(genes_df))
print('Number of genes with BLAST hits:', blast_ggkbase_df.query_id.nunique())
print('Number of unique targets:', blast_ggkbase_df.target_id.nunique())

apply_filters: 19182 entries removed by large_length_difference.
apply_filters: 15330 entries removed by low_query_coverage.
apply_filters: 6757 entries removed by bz_12_gene.
Number of query genes: 402
Number of genes with BLAST hits: 352
Number of unique targets: 18997


## Generating a3m-formatted alignments

We obtained the amino acid sequences for the 

In [ ]:
tmp_dir = '../data/genes/mmseqs/tmp'

sensitivity = 9.5 
coverage_mode = 5
min_seq_identity = 0.98

In [ ]:
def dereplicate(path, output_dir=None, query_gene_ids:list=None, min_seq_identity:float=0.95, min_coverage=0.95, coverage_mode:int=5):
    '''

    :param path: The path to a FASTA file containing the sequences to dereplicate. 
    :param output_dir: The directory where all MMseqs output files will be written. 
    :param query_gene_ids: The IDs of sequences which will be input to AlphaFold, and therefore require an af3 MSA where they are the 
        reference sequence (i.e. in the first line of the file). These genes must be included in the dereplicated dataset.
    :param min_seq_identity: The minimum sequence identity for MMseqs clustering. 
    :param min_coverage: The minimum coverage mode for MMseqs clustering. 
    :param coverage_mode: The coverage mode, which specifies how coverage will be computed.
    '''

    name = os.path.basename(path).split('.')[0]
    output_path = os.path.join(output_dir, name) # The file root name; this does not have a file extension. 

    cmd = f'mmseqs easy-cluster {path} {output_path} {tmp_dir} --cov-mode {coverage_mode} -c {min_coverage} --min-seq-id {min_seq_identity}'
    print('dereplicate:', cmd)
    subprocess.run(cmd, shell=True, check=True, stdout=subprocess.DEVNULL)

    # Load the mmseqs clusters and select representatives, ensuring the MSA queries are included in the final set. 
    cluster_df = pd.read_csv(output_path + '_cluster.tsv', names=['rep_gene_id', 'gene_id'], sep='\t')
    cluster_df['in_query_gene_ids'] = cluster_df.gene_id.isin(query_gene_ids)
    cluster_df = cluster_df.sort_values('in_query_gene_ids', ascending=False)
    # If two queries end up in the same cluster, this can cause problems, because one will be dropped despite the sorting step.
    # Therefore, we take the union of the query IDs and de-replicated sequences to ensure all query gene IDs are included. 
    cluster_df = cluster_df[~cluster_df.rep_id.duplicated(keep='first') | cluster_df.in_query_ids]

    assert np.all(np.isin(query_gene_ids, cluster_df.gene_id.unique())), f'msa_dereplicate: Some query IDs are missing from the set of dereplicated sequences.'
    print(f'dereplicate: {len(cluster_df)} sequences remaining after dereplication.')

    reps_df = FASTAFile.from_file(path).to_df() # Load the input sequences. 
    reps_df = reps_df.loc[cluster_df.gene_id.unique()].copy() # Get the representative sequences. 
    reps_df = reps_df[~reps_df.index.duplicated(keep='first')].copy() # Ensure there are no duplicates (happening sometimes, not sure why).
    FASTAFile.from_df(reps_df).write(os.path.join(output_dir, f'{name}_reps.faa'))

    # Clean up the output files. 
    for extension in ['_cluster.tsv', '_rep_seq.fasta', '_all_seqs.fasta']:
        os.remove(output_path + extension)

    return output_path + '_reps.faa'

In [4]:
def get_custom_msas(path:str, output_dir:str=None, query_gene_ids:list=None, database_dir:str=None, sensitivity=9.5, num_iterations:int=5, **kwargs):
    '''Use MMseqs align utilities to construct a3m-format alignment files for each sequence in the input FASTA file. The pipeline is as follows:
        (1) Use the input FASTA file to construct an MMseqs database. 
        (2) Generate a search result database by querying the input database against itself. 
        (3) Construct full alignments using the results of the search. 
        (4) Convert the alignments to MSAs. 
        (5) Unpack the alignment output info a3m files compatible with AlphaFold; there will be one file per query.
        
    :param path: The path to the FASTA file containing all sequences to use in MSA construction.
    :param output_dir: The directory where all output files will be deposited. 
    :param query_gene_ids: The IDs of sequences which will be input to AlphaFold, and therefore require an af3 MSA where they are the 
        reference sequence (i.e. in the first line of the file). 
    :param sensitivity: The search sensitivity for the initial MMseqs search step. 
    :param num_iterations: The number of iterations for the MMseqs search step. 
    '''
    database_name = 'tmp' # Just use this as a throwaway database name. 
    database_path = os.path.join(database_dir, database_name)
    databases = Databases(database_path, f'{database_path}.out', f'{database_path}.aln', f'{database_path}.msa')

    kwargs = {'shell':True, 'check':True, 'stdout':subprocess.DEVNULL}
    cmds = [f'mmseqs createdb {path} {databases.input}']
    cmds += [f'mmseqs search {databases.input} {databases.input} {databases.output_search} {tmp_dir} -s {sensitivity} --num-iterations {num_iterations}']
    cmds += [f'mmseqs align {databases.input} {databases.input} {databases.output_search} {databases.output_align}']
    cmds += [f'mmseqs result2msa {databases.input} {databases.input} {databases.output_align} {databases.output_msa}']
    cmds += [f'mmseqs unpackdb {databases.output_msa} {output_dir} --unpack-suffix .a3m']

    for cmd in cmds:
        print('get_custom_msas:', cmd)
        subprocess.run(cmd, **kwargs)

    msa_paths = list()

    # The unpacked database files just have numerical IDs, so want to rename according to their actual gene IDs. 
    # This is done by mapping the numerical ID to the original gene ID using the lookup table associated with the input database. 
    n_to_gene_id_map = pd.read_csv(databases.input + '.lookup', sep=r'\s+', names=['num', 'gene_id'], index_col=0, usecols=[0, 1]).gene_id.to_dict()

    # Function to determine which MSAs to keep, which is determined by whether or not the alignment file matches a query gene ID. 
    pattern = '|'.join([gene_id.replace('.', r'\.') for gene_id in query_gene_ids]) if (query_gene_ids is not None) else r'.*'
    keep_msa = lambda msa_path : re.search(pattern, msa_path) is not None 

    for file_name in os.listdir(output_dir):
        if not re.match(r'\d+\.a3m', file_name):
            continue 
        n = int(re.match(r'(\d+)\.a3m', file_name).group(1))
        old_path = os.path.join(output_dir, file_name)
        new_path = os.path.join(output_dir, f'{n_to_gene_id_map[n]}.a3m')
        
        if keep_msa(new_path):
            msa_paths.append(new_path)
            subprocess.run(f'mv {old_path} {new_path}', shell=True, check=True)
        else:
            os.remove(old_path)

    print(f'get_custom_msas: Generated {len(msa_paths)} total MSAs.')

In [ ]:
genes_df = FASTAFile.from_file('0-structures-custom_msa-genes_prodigal.faa').to_df()
genes_df = genes_df[~genes_df.index.str.contains('bz_12')].copy()
query_gene_ids = genes_df.index.values 

blast_ggkbase_df = FASTAFile.from_file('../data/genes/blast/ggkbase/genes_prodigal.faa').to_df()
blast_ggkbase_df.index = [gene_id.split('|')[0] for gene_id in blast_ggkbase_df.index] # Clean up the ggKbase indices. 

database_dir = '../data/genes/mmseqs/tmp/db'
os.makedirs(database_dir, exist_ok=True)

database_path = '../data/genes/mmseqs/custom_msas_database.faa' # Input FASTA file for the database sequences.
database_df = pd.concat([genes_df, blast_ggkbase_df])
FASTAFile.from_df(database_df).write(database_path)

# database_path = msa_dereplicate(database_path, query_gene_ids=query_gene_ids, output_dir='../data/genes/mmseqs/', min_seq_identity=0.95, min_coverage=0.95, coverage_mode=5)
# get_custom_msas(database_path, query_gene_ids=query_gene_ids, output_dir='../data/genes/mmseqs/msa/', num_iterations=5, sensitivity=9.5, database_dir=database_dir)


In [33]:
alphafold_input_dir = f'../data/genes/alphafold/af_input/genes_prodigal-custom_msa'
alphafold_output_dir = f'../data/genes/alphafold/af_output/genes_prodigal-custom_msa' 
os.makedirs(alphafold_input_dir, exist_ok=True)

# Configure the AlphaFold3 job, using fewer diffusion samples and recycles to reduce runtime. 
alphafold_params = dict()
alphafold_params['--num_diffusion_samples'] = 5 # Number of diffusion samples to generate, 5 by default. 
alphafold_params['--num_recycles'] = 5 # Number of recycles to use during inference (refinement steps). 10 by default, dropped here to mitigate runtime. 

In [30]:
msas = get_msas_unpaired(genes_df.index)

print(f'Number of genes:', len(genes_df))
print('Number of genes with custom MSAs:', len(msas))


Number of genes: 402
Number of genes with custom MSAs: 402


In [34]:
for row in genes_df.itertuples():

    input = AlphaFoldInputFile(row.Index, dialect='alphafold3', num_seeds=3, version=2) # Use 3 seeds rather than the default 5 to reduce runtime.
    input.add_seq(row.seq, paired_msa='', unpaired_msa=msas.get(row.Index))
    input.write(os.path.join(alphafold_input_dir, f'{row.Index}.json'))

cmd = [f'run_alphafold3 --input_dir=/root/af_input/{os.path.basename(alphafold_input_dir)}/ --model_dir=/root/models --db_dir=/root/public_databases --output_dir=/root/af_output/{os.path.basename(alphafold_input_dir)}/']
cmd += [f'{param} {value}' for param, value in alphafold_params.items()]
cmd = ' '.join(cmd)
cmd = f'sbatch --partition gpu --gpus 1 --wrap "{cmd}"'
print(cmd)

sbatch --partition gpu --gpus 1 --wrap "run_alphafold3 --input_dir=/root/af_input/genes_prodigal-custom_msa/ --model_dir=/root/models --db_dir=/root/public_databases --output_dir=/root/af_output/genes_prodigal-custom_msa/ --num_diffusion_samples 5 --num_recycles 5"
